# Global PCA for Industrial Visual Anomaly Detection

**CS 663 — Digital Image Processing | MVTec AD**

This notebook fits a global PCA model using only defect-free training images. Each image is represented by one flattened grayscale vector. The model reconstructs test images from a retained principal subspace, and squared reconstruction residuals form the anomaly maps.

The experiment compares retained variance, residual-map smoothing and image-score aggregation. All settings are fixed before test evaluation.

## 1. Setup

The dataset root must directly contain the 15 MVTec category folders. In Colab, mount Drive before running this cell and set `DATA_ROOT` to the shared dataset directory.

In [ ]:
from pathlib import Path
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter
from sklearn.metrics import average_precision_score, roc_auc_score

from src.data.dataset import MVTecDataset, available_categories
from src.data.preprocessing import prepare_image, prepare_mask
from src.evaluation import evaluate_category, image_score
from src.pca.global_pca import GlobalPCA

DATA_ROOT = Path("/content/drive/MyDrive/CS663_Shared/datasets/mvtec_anomaly_detection")
if not DATA_ROOT.is_dir():
    DATA_ROOT = Path("data/mvtec_anomaly_detection")

IMAGE_SIZE = 256
SEED = 42
VAL_FRACTION = 0.10
RUN_ALL_CATEGORIES = True
SMOKE_TEST_CATEGORY = "bottle"
RETAINED_VARIANCES = [0.90, 0.95, 0.98]
SMOOTHING_SIGMAS = [0.0, 0.75]
IMAGE_SCORE_MODES = ["max", "top_1pct"]

EXPECTED_CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATA_ROOT}")

available = available_categories(DATA_ROOT)
if RUN_ALL_CATEGORIES:
    missing = sorted(set(EXPECTED_CATEGORIES) - set(available))
    if missing:
        raise FileNotFoundError(f"Missing categories: {missing}")
    CATEGORIES = EXPECTED_CATEGORIES
else:
    if SMOKE_TEST_CATEGORY not in available:
        raise FileNotFoundError(f"Category not found: {SMOKE_TEST_CATEGORY}")
    CATEGORIES = [SMOKE_TEST_CATEGORY]

print("Dataset:", DATA_ROOT)
print("Categories:", CATEGORIES)
print("Image size:", IMAGE_SIZE)

## 2. Preprocessing and evaluation helpers

Images are converted to grayscale, scaled to [0,1] and resized to 256 by 256. Masks use max pooling when downsampled to preserve small defect regions. The image threshold is calibrated using normal validation images only.

In [ ]:
def load_split(category, split):
    return MVTecDataset(
        DATA_ROOT,
        category,
        split=split,
        val_fraction=VAL_FRACTION,
        seed=SEED
    )


def prepare_images(dataset):
    return np.stack([
        prepare_image(dataset[index]["image"], IMAGE_SIZE)
        for index in range(len(dataset))
    ])


def prepare_test_data(dataset):
    images = []
    masks = []
    labels = []
    for index in range(len(dataset)):
        sample = dataset[index]
        images.append(prepare_image(sample["image"], IMAGE_SIZE))
        masks.append(prepare_mask(sample["mask"], IMAGE_SIZE, mode="max"))
        labels.append(sample["label"])
    return np.stack(images), np.asarray(labels, dtype=np.uint8), np.stack(masks)


def smooth_maps(maps, sigma):
    if sigma == 0:
        return maps
    return gaussian_filter(maps, sigma=(0, sigma, sigma)).astype(np.float32)


def evaluate_setting(validation_maps, test_labels, test_maps, test_masks, score_mode):
    validation_scores = np.asarray([
        image_score(anomaly_map, score_mode) for anomaly_map in validation_maps
    ])
    threshold = float(np.quantile(validation_scores, 0.95, method="higher"))
    test_scores = np.asarray([
        image_score(anomaly_map, score_mode) for anomaly_map in test_maps
    ])
    metrics = evaluate_category(
        test_labels, test_scores, test_masks, test_maps, threshold
    )
    return metrics, threshold


toy_mask = np.zeros((2, 16, 16), dtype=bool)
toy_mask[0, 5:10, 5:10] = True
toy_maps = np.zeros((2, 16, 16), dtype=np.float32)
toy_maps[toy_mask] = 1.0
assert np.isclose(
    __import__("src.evaluation", fromlist=["compute_aupro"]).compute_aupro(toy_mask, toy_maps),
    1.0
)
print("Evaluation sanity check passed.")

## 3. Global PCA model

The centered training matrix is decomposed using economy SVD. This obtains the principal directions of the pixel covariance matrix without explicitly constructing a large pixel-by-pixel covariance matrix.

The retained components are selected by cumulative explained variance. Projection and reconstruction follow the standard PCA equations.

In [ ]:
def fit_full_pca(training_images):
    return GlobalPCA(
        n_components="all",
        method="svd",
        residual="squared",
        ddof=0
    ).fit(training_images)


def retained_component_count(model, variance):
    cumulative = np.cumsum(model.eigenvalues_all_) / model.total_variance_
    return int(np.searchsorted(cumulative, variance) + 1)


def anomaly_maps(model, images):
    return model.anomaly_map(images).astype(np.float32)


random_images = np.random.default_rng(SEED).normal(size=(8, 1, 16, 16)).astype(np.float32)
sanity_model = GlobalPCA(n_components="all", method="svd").fit(random_images)
sanity_reconstruction = sanity_model.reconstruct(random_images)
assert sanity_reconstruction.shape == random_images.shape
assert np.isfinite(sanity_model.anomaly_map(random_images)).all()
print("Global PCA sanity check passed.")

## 4. Run the category experiments

For each category, one full PCA basis is fitted using normal training images. The same ordered basis is truncated for the three retained-variance settings. Validation and test maps are generated for each setting, then the smoothing and image-scoring ablations are evaluated.

In [ ]:
all_results = []
preview = None

for category in CATEGORIES:
    start_time = time.perf_counter()
    train_dataset = load_split(category, "train")
    validation_dataset = load_split(category, "val")
    test_dataset = load_split(category, "test")

    if any(len(dataset) == 0 for dataset in (train_dataset, validation_dataset, test_dataset)):
        raise ValueError(f"An empty split was found for {category}.")
    if not all(sample[2] == "good" for sample in train_dataset.samples):
        raise ValueError("PCA fitting data must contain train/good images only.")
    if not all(sample[2] == "good" for sample in validation_dataset.samples):
        raise ValueError("Validation data must contain normal images only.")

    training_images = prepare_images(train_dataset)
    validation_images = prepare_images(validation_dataset)
    test_images, test_labels, test_masks = prepare_test_data(test_dataset)

    full_model = fit_full_pca(training_images)
    validation_maps_full = anomaly_maps(full_model, validation_images)
    test_maps_full = anomaly_maps(full_model, test_images)

    for variance in RETAINED_VARIANCES:
        component_count = retained_component_count(full_model, variance)
        model = full_model.truncate(component_count)
        actual_variance = float(model.explained_variance_ratio_.sum())
        validation_maps_raw = anomaly_maps(model, validation_images)
        test_maps_raw = anomaly_maps(model, test_images)

        for sigma in SMOOTHING_SIGMAS:
            validation_maps = smooth_maps(validation_maps_raw, sigma)
            test_maps = smooth_maps(test_maps_raw, sigma)

            for score_mode in IMAGE_SCORE_MODES:
                metrics, threshold = evaluate_setting(
                    validation_maps, test_labels, test_maps, test_masks, score_mode
                )
                all_results.append({
                    "category": category,
                    "retained_variance_target": variance,
                    "retained_variance_actual": actual_variance,
                    "components": component_count,
                    "smoothing_sigma": sigma,
                    "image_score": score_mode,
                    "training_images": len(train_dataset),
                    "validation_images": len(validation_dataset),
                    "test_images": len(test_dataset),
                    "validation_threshold": threshold,
                    **metrics
                })

        if preview is None and np.any(test_labels == 1):
            first_anomaly = int(np.flatnonzero(test_labels == 1)[0])
            preview = (
                test_images[first_anomaly, 0],
                test_maps_raw[first_anomaly],
                smooth_maps(test_maps_raw[first_anomaly: first_anomaly + 1], 0.75)[0],
                test_masks[first_anomaly]
            )

    elapsed = time.perf_counter() - start_time
    print(f"{category}: completed in {elapsed:.1f} seconds")

results = pd.DataFrame(all_results)
metric_columns = [
    "image_auroc", "image_ap", "image_precision", "image_recall", "image_f1",
    "pixel_auroc", "pixel_ap", "aupro_30pct_fpr"
]
summary = (
    results.groupby(
        ["retained_variance_target", "smoothing_sigma", "image_score"]
    )[metric_columns]
    .mean()
    .reset_index()
)

display(results.round(4))
display(summary.round(4))

output_dir = Path("results/tables")
output_dir.mkdir(parents=True, exist_ok=True)
results.to_csv(output_dir / "global_pca_categories.csv", index=False)
summary.to_csv(output_dir / "global_pca_summary.csv", index=False)
print("Result tables saved in", output_dir)

## 5. Visual inspection

The example below compares a test image, its raw reconstruction residual, the smoothed residual and its ground-truth mask. The visualization is qualitative; quantitative metrics are computed from unnormalized anomaly scores.

In [ ]:
if preview is not None:
    figure, axes = plt.subplots(1, 4, figsize=(15, 4))
    titles = ["Test image", "Raw residual", "Smoothed residual", "Ground-truth mask"]
    images = [preview[0], preview[1], preview[2], preview[3]]
    for axis, image, title in zip(axes, images, titles):
        axis.imshow(image, cmap="inferno" if "residual" in title else "gray")
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plot_dir = Path("results/plots")
    plot_dir.mkdir(parents=True, exist_ok=True)
    figure.savefig(plot_dir / "global_pca_example.png", dpi=180, bbox_inches="tight")
    plt.show()

## 6. Interpretation

- Retained variance controls the number of principal directions. More components improve normal-image reconstruction but can also reconstruct defect structure.
- Gaussian smoothing can suppress isolated residual noise and produce more spatially coherent maps. It can also blur small defects.
- Top-one-percent aggregation reduces dependence on a single extreme pixel compared with max scoring. It may dilute very small defects.
- Image AUROC and average precision evaluate ranking independently of the chosen threshold. Precision, recall and F1 depend on the validation-derived threshold.
- AUPRO summarizes connected-region overlap up to a specified pixel false-positive rate.

The ablation settings are reported together. A configuration should only be described as better if the complete category results support that conclusion. Global PCA remains a global, position-sensitive linear model and may struggle with local texture changes or object misalignment.

## 7. References

- CS 663 Digital Image Processing lecture material, PCA section.
- Bergmann et al. (2019), *MVTec AD — A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection*, CVPR. [Dataset page](https://www.mvtec.com/research-teaching/datasets/mvtec-ad).
- Jolliffe and Cadima (2016), *Principal component analysis: a review and recent developments*. [DOI](https://doi.org/10.1098/rsta.2015.0202).